# पाठ १८: क्रिप्टोग्राफिक रसीदहरूसँग AI एजेन्टहरूलाई सुरक्षित बनाउने

## ह्यान्ड्स-अन नोटबुक

यो नोटबुक चार अभ्यासहरू मार्फत जान्छ:

१. कुनै एजेन्ट टुल कलको लागि **पहिलो रसीदमा हस्ताक्षर गर्नुहोस्** र त्यसलाई प्रमाणीकरण गर्नुहोस्।
२. **रसीदमा छेड़छाड गर्नुहोस्** र प्रमाणीकरण असफल हुने हेर्नुहोस्।
३. **तीन-रसीद श्रृंखला निर्माण गर्नुहोस्** र श्रृंखलाको अखण्डता पुष्टि गर्नुहोस्।
४. **वस्तुतः प्रत्येक कार्यले रसीद निकाल्ने गरी Microsoft एजेन्ट फ्रेमवर्क टुल कललाई र्‍याप गर्नुहोस्।**

सबै क्रिप्टोग्राफिक प्रिमिटिभहरू राम्ररी व्यवस्थापन गरिएका पुस्तकालयहरूबाट आयात गरिएको छ (`pynacl` Ed25519 का लागि, `jcs` RFC 8785 कैनोनिकल JSON का लागि, र Python मानक पुस्तकालयबाट `hashlib` SHA-256 का लागि)। रसीद तर्क आफैँ साधारण Python हो जुन तपाईंले पढ्न र संशोधन गर्न सक्नुहुन्छ।

अर्डरमा सेलहरू चलाउनुहोस्। हरेक भाग छोटो र स्व-निहित छ।


## सेटअप

दुई निर्भरताहरू स्थापना गर्नुहोस्। दुबैमा अनुमति दिने लाइसेन्सहरू छन् (Apache-2.0 / MIT)।


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## सहायक उपयोगिताहरू

यी दुई सहायकहरूले बेस64url इन्कोडिंग (प्याडिंग बिना) र मनग्य वस्तुहरूको SHA-256 हैसिङ् सम्हाल्छन्। तिनीहरूले नोटबुकको बाँकी भागलाई रिसिप्ट तर्कमा केन्द्रित राख्छन्।


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## खण्ड १: तपाईँको पहिलो रसिदमा हस्ताक्षर गर्नुहोस्

कल्पना गर्नुहोस् हाम्रो एजेन्ट **Contoso Travel** ले ग्राहकको लागि सिड्नीबाट लॉस एन्जलससम्मको उडानहरू हेरेको छ। हामी भविष्यमा एक अडिटरले फरक पिन गरिएको जारीकर्ता कुञ्जी अन्तर्गत यो उपकरण कलको गेटवेको वक्तव्य प्रमाणित गर्न सकून् भनी यसलाई रेकर्ड गर्न चाहन्छौं।

### चरण १.१: हस्ताक्षर कुञ्जी सिर्जना गर्नुहोस्

उत्पादनमा, एजेन्टको हस्ताक्षर कुञ्जी हार्डवेयर सुरक्षा मोड्युल (HSM), Azure Key Vault, वा यस्तै सुरक्षित स्टोरमा राखिन्छ। यस पाठको लागि हामी मेमोरीमा नयाँ कुञ्जी सिर्जना गर्छौं।


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### चरण १.२: रसीद पेलोड बनाउनुहोस्

पेलोडमा हामीले रसीदमा प्रमाणित गर्न चाहेको सबै कुरा समावेश छ: कसले काम गर्‍यो, कुन उपकरण, कुन तर्कसँग, के फर्कियो, कुन नीतिमा, र कहिले। हामी तर्क र परिणामलाई इनलाइन समावेश नगरी तिनीहरूलाई ह्यास गर्छौं ताकि रसीदले संवेदनशील सामग्री चुहावट नगरोस्।


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### चरण 1.3: प्राप्ति मा हस्ताक्षर गर्नुहोस् र संयोजन गर्नुहोस्

तीन चरणहरू:

1. JCS प्रयोग गरेर पेलोडलाई canonicalize गर्नुहोस् ताकि दुई कार्यान्वयनहरूले समान तार्किक प्राप्ति उत्पादन गर्दा बाइट-परिचयात्मक बाइटहरू उत्पादन गर्छन्।
2. canonical बाइटहरूलाई सिधै Ed25519 निजी कुञ्जीले हस्ताक्षर गर्नुहोस्। PureEdDSAले सन्देशलाई आन्तरिक रूपमा ह्यास गर्छ, त्यसैले अतिरिक्त प्रि-ह्यासले प्रोटोकललाई परिवर्तन गर्नेछ।

त्यसपछि हस्ताक्षरलाई मूल पेलोडमा संलग्न गरिन्छ ताकि अन्तिम प्राप्ति उत्पादन होस्।


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### चरण १.४: रसिदको प्रमाणीकरण गर्नुहोस्

प्रमाणीकरणले प्रक्रियालाई उल्टो गर्छ। हामी हस्ताक्षर हटाउँछौं, क्यानोनिकल बाइटहरू पुनः गणना गर्छौं, र जारीकर्ताको सार्वजनिक कुञ्जीसँग हस्ताक्षर जाँच गर्छौं।

प्रमाणीकरण अनलाइन बाहिर हुन्छ: कल गर्ने सेवा वा नेटवर्क पहुँच हुँदैन। यसले रसिद बाहेक एउटा कुरा आवश्यक छ: जारीकर्ताको सार्वजनिक कुञ्जी, जुन अडिटरले पहिले नै विश्वास गरेको च्यानल मार्फत प्राप्त हुन्छ, जस्तै जारीकर्ताले प्रकाशित गरेको JWK सेट वा पूर्वनिर्धारित मूल्य। रसिद भित्रको `public_key` क्षेत्रले त्यो भूमिका निभाउन सक्दैन, किनकि जसले रसिद लेख्छ उसैले त्यो क्षेत्र पनि लेख्छ। खण्ड २ ले यसले रोक्ने नक्कली कार्य देखाउँछ।


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


तपाईंले `Receipt is valid: True`, `Pre-hashed receipt valid: False` र `Malformed receipts rejected: True` देख्नुपर्छ। सकारात्मक केसले सिधा-JCS हस्ताक्षर गर्ने बाटो काम गर्छ भनेर प्रमाणित गर्दछ; नकारात्मक नियन्त्रणले हस्ताक्षर-क्षेत्र नियमलाई व्याख्या नगरी prose को रूपमा छोड्नुको सट्टा कार्यान्वयनयोग्य बनाउँछ; अन्तिम लाइनले malformed इनपुटले अपवाद उठाउने सट्टा `False` फिर्ता गर्छ देखाउँछ।


## भाग २: रसिदमा छेडछाड गर्नुहोस्

रसिदहरूको पूरा उद्देश्य नै हो कि ती छेडछाड प्रमाणित हुन्छन्। यसलाई प्रमाणित गरौं।

हामी रसिदको ठीक एक पात्रो मात्र परिमार्जन गर्नेछौं र प्रमाणीकरण असफल हुने हेर्नेछौं।


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### के भयो?

जब हामीले `policy_id` परिवर्तन गर्यौं, क्यानोनिकल बाइटहरू परिवर्तन भए। हस्ताक्षर (जे क्यानोनिकल बाइटहरूमाथि रहेको थियो) अब मेल खाँदैन। प्रमाणीकरणले सही रूपमा `False` फर्काउँछ।

कुनै पनि तरिकाले साइन गरिएको फिल्ड परिवर्तन गरेर, जसले `signature` वस्तुको बाहिर पर्ने कुरा हो, र अझै पनि रसीद जारीकर्ताको सार्वजनिक कुञ्जीसँग प्रमाणीकरण हुने सम्भावना छैन, जबसम्म आक्रमणकारीसँग जारीकर्ताको निजी कुञ्जी नहुन्। `signature` वस्तु आफैले साइन गरिएको छैन: `sig` हस्ताक्षर हो, र `verify_receipt` ले `public_key` लाई बेवास्ता गर्छ, त्यसैले त्यो फिल्ड मात्र सम्पादन गर्दा केही परिवर्तन हुँदैन। जबसम्म निजी कुञ्जी कुञ्जी त्रिेकामा हुन्छ र प्रमाणीकरण गर्नेहरूले जारीकर्ताको स्वतन्त्र रूपमा प्रकाशित सार्वजनिक कुञ्जी प्रयोग गर्छन्, साइन गरिएको फिल्डमा परिवर्तन लुकाउन असम्भव हुन्छ। त्यो अवस्था महत्वपूर्ण छ: अर्को सेलमा देखाइएको छ परिवर्तन गरिएको रसीद जुन फरक कुञ्जीसँग पूर्ण रूपमा प्रमाणीकरण हुन्छ।

आफैं प्रयास गर्नुहोस्: माथि सेलमा `tool_name` वा `agent_id` वा `timestamp` परिवर्तन गरेर पुन: चलाउनुहोस्। साइन गरिएको फिल्डमा हरेक परिवर्तन अवैध रसीद उत्पादन गर्छ।


### हस्ताक्षरले पक्रन नसक्ने छेउछाउको परिवर्तन

माथिको आक्रमणकारीले पुनः हस्ताक्षर गरेन। एक धेरै सावधान व्यक्तिले गर्न सक्छ: रसिद परिवर्तन गर्छ,
आफूले सिर्जना गरेको कुञ्जीले यसमा हस्ताक्षर गर्छ, र त्यो कुञ्जीलाई
`public_key` फिल्डमा राख्छ। परिणामस्वरूप रसिदको हस्ताक्षर यसको सामग्रीको लागि पूर्ण रूपमा वास्तविक हुन्छ।


यसलाई चलाउनुहोस् र कुन दुई मध्ये कस्तो जाँचले यसलाई पक्रन्छ हेर्नुहोस्।


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


तपाईंले `True` अनि त्यसपछि `False` देख्नु पर्नेछ। ती तीनवटा `assert` लाइनहरूले त्यो
जाँच बनाउँछन्: यदि फ्रोजरी कहिल्यै जारीकर्ताको कुञ्जीसँग प्रमाणित भयो भने सो सेलले त्रुटि उठाउँछ।


फ्रोजरी आन्तरिक रूपमा सुसंगत छ। यसको हस्ताक्षर साँच्चिकै यसको
सामग्री र यसले बोकेको कुञ्जीसँग मेल खान्छ, त्यसैले जसले रसिदबाट कुञ्जी पढ्छ, त्यसले यसलाई मान्य भनेर रिपोर्ट गर्छ। रसिदमा केही पनि गलत देखिंदैन,
किनभने त्यसमा केही पनि असंगत छैन।


यसलाई समात्ने एक मात्र कुरा भनेको एक यस्तो कुञ्जी विरुद्ध जाँच गर्नु हो जुन आक्रमणकारीले प्रभाब पार्न सकेन।
यो कारणले गर्दा `verify_receipt` ले कुञ्जीलाई एक दाबीको रूपमा लिन्छ: एक
फंक्शन जसको हस्ताक्षर `verify_receipt(receipt)` हो सुरक्षित रूपमा लेख्न सकिदैन,
किनभने एक मात्र कुञ्जी जुन यसले पाउन सक्छ त्यो आक्रमणकर्ताले दिएको हो।

एउटै रसिद `sample_receipts/04_resigned_receipt.json` को रूपमा सुरक्षित गरिन्छ यदि तपाईं
यसलाई नोटबुक बाहिर निरीक्षण गर्न चाहनुहुन्छ भने।


## खण्ड ३: रसिदहरूलाई सँगसँगै जोड्नुहोस्

एकल रसिदले एक क्रियालाई संरक्षण गर्छ। धेरै एजेन्टहरूले धेरै क्रियाहरू गर्छन्। सम्पूर्ण श्रृंखलालाई छेड़छाड़-प्रमाणित बनाउनका लागि, हामी प्रत्येक रसिदलाई अघिल्लो रसिदको ह्यास नयाँ रसिदको पेलोडमा समावेश गरेर अघिल्लो रसिदसँग जोड्छौं।

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

यदि कसैले प्रस्तुत गरिएको श्रृंखलाभित्रको रसिद हटाउन वा पुनःक्रमबद्ध गर्नुभयो भने, अर्को लिङ्क टुट्छ। ट्रंकनसन वा साइनर-नियन्त्रित सिफिक्स पुनर्लेखन पत्ता लगाउनको लागि हालको हेड पिन गरिएको, बाह्य चेकप्वाइन्ट, वा प्रमाणित जनसंख्या प्रतिबद्धता आवश्यक पर्छ।


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


अब मध्य रसीदमा छेडखानी गरेर श्रृंखला तोड्नुहोस् र पुनः प्रमाणीकरण गर्नुहोस्। छेडखानी गरिएको रसीदले यसको हस्ताक्षर जाँचमा असफल हुन्छ, र अर्को रसीदले यसको श्रृंखला-लिंक जाँचमा असफल हुन्छ (किनभने यसको `previous_receipt_hash` अब संशोधित मध्य रसीदको ह्याससँग मेल खाँदैन)।


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


रसिद ० अझै प्रमाणित गर्छ (यो संशोधित गरिएको थिएन र निर्भर गर्न कुनै पूर्वसर्ती थिएन)। रसिद १ यसको हस्ताक्षर जाँचमा असफल हुन्छ किनभने हामीले `tool_args_hash` परिवर्तन गरेका थियौं। रसिद २ यसको चेन-लिंक जाँचमा असफल हुन्छ किनभने यसको `previous_receipt_hash` मूल (अब संशोधित) रसिद १ सँग गणना गरिएको थियो।  

यदि आक्रमणकारीले संशोधित रसिद १ लाई पुनःहस्ताक्षर पनि गरे भने, नयाँ हस्ताक्षर जारीकर्ताको कुञ्जी विरुद्ध असफल हुन्छ जबसम्म उनीहरूसँग जारीकर्ताको निजी कुञ्जी हुँदैन (आफ्नो कुञ्जीसँग पुनःहस्ताक्षर गर्दा असफल हुन्छ, जसरी खण्ड २ मा देखाइएको छ), र रसिद २ मा चेन-लिंक मेल नखाने कुराले समेत छेडछाड उजागर गर्थ्यो। परिवर्तन लुकाउन आक्रमणकारीले परिवर्तन को स्थानदेखि सबै रसिदहरू पुनःहस्ताक्षर गर्नुपर्ने हुन्थ्यो, जुन जारीकर्ताको निजी कुञ्जीको कब्जा आवश्यक छ।  


## अनुभाग 4: एक एजेन्ट उपकरण कललाई रिसिप्ट साइनिङसँग र्याप गर्ने

वास्तविक कार्यान्वयनमा, तपाईंले प्रत्येक एजेन्ट लेखकलाई `make_receipt` कल गर्न सम्झन नदिइकनै राख्न चाहनुहुन्छ। तपाईं चाहनुहुन्छ कि प्रत्येक उपकरण कलको लागि रिसिप्ट साइनिङ स्वचालित होओस्।

यहाँ सबैभन्दा साधारण ढाँचा छ: एउटा र्यापर क्लास जसले कुनै पनि कल गर्न सकिने उपकरण कार्यलाई लिएर यसको रिसिप्ट-प्रयोग गर्ने संस्करण फर्काउँछ। यसले माइक्रोसफ्ट एजेन्ट फ्रेमवर्क (`agent_framework.foundry`) लगायत कुनै पनि एजेन्ट फ्रेमवर्कमा अनुकूल हुन्छ।

यदि तपाईंले माइक्रोसफ्ट फाउन्ड्री प्रोजेक्ट सेटअप गर्नुभएको छैन भने पनि तलको लोक्कले यो ढाँचा देखाउँछ।


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### माइक्रोसफ्ट एजेन्ट फ्रेमवर्कसँग एकीकरण

माथि दिएको `ReceiptedTool` र्‍यापर फ्रेमवर्क-स्वतन्त्र छ। माइक्रोसफ्ट एजेन्ट फ्रेमवर्क प्रयोग गरी बनाइएको एजेन्ट भित्र यसलाई प्रयोग गर्न, र्‍याप गरिएको फङ्क्शनलाई टूलको रूपमा दर्ता गर्नुस्। एउटा स्केच (तपाईंले नक्कलीलाई वास्तविक माइक्रोसफ्ट फाउन्ड्री टूल दर्तासँग प्रतिस्थापन गर्नुहुनेछ):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

एजेन्ट फ्रेमवर्कलाई रसिदहरूको बारेमा केही जान्न आवश्यक छैन। रसिद हस्ताक्षर टूल वरिपरि र्‍याप गरिएको छ, फ्रेमवर्कमा जोडिएको छैन। यसरी तपाईंले एजेन्टलाई पुन:लेखन नगरी रहेको एजेन्ट कोडमा प्रमाण थप्न सक्नुहुन्छ।


## पुनरावलोकन र स्ट्रेच चुनौती

तपाईंले:

- एक Ed25519 कुञ्जी जोडी उत्पन्न गर्नुभएको छ।
- एक एजेन्ट उपकरण कलको लागि रसिद तयार र हस्ताक्षर गर्नुभएको छ।
- रसिदलाई अफलाइन रूपमा केवल जारीकर्ताको सार्वजनिक कुञ्जी प्रयोग गरी सत्यापित गर्नुभएको छ, जुन रसिदबाट स्वतन्त्र रूपमा प्राप्त गरिएको हो।
- रसिदसँग छेडछाड गर्नुभयो र सत्यापन असफल भएको अवलोकन गर्नुभयो।
- तीन वटा रसिदहरूको ह्यास-चेन गरिएको अनुक्रम तयार गर्नुभयो।
- चेनको बीच भागमा छेडछाड गर्नुभयो र दुवै हस्ताक्षर असफल र चेन-लिंक असफलता देख्नुभयो।
- एउटा उपकरण कार्यलाई स्वचालित रसिद हस्ताक्षरको साथ आवरण गर्नुभयो।

**स्ट्रेच चुनौती।** रसिद स्कीमामा `request_id` क्षेत्र थप्नुहोस् (वितरित ट्रेसिङको लागि UUID)। यसलाई समावेश गर्न `make_receipt` अद्यावधिक गर्नुहोस्, र रसिदहरू अन्त्यसम्म सत्यापित भइरहेका छन् भनेर पुष्टि गर्नुहोस्। त्यसपछि हस्ताक्षर पछि क्षेत्रलाई संशोधित गरी पुष्टि गर्नुहोस् कि सत्यापन असफल हुन्छ। यसले तपाईंलाई क्यानोनिकल इन्कोडिङको प्रत्येक बाइटले हस्ताक्षरमा कसरी योगदान पुर्याउँछ भनी आन्तरिक रूपमा बुझ्न बाध्य पार्छ।

**महत्वपूर्ण सीमा।** पिन गरिएको कुञ्जी र चेन सन्दर्भ अन्तर्गत, रसिदहरूले हस्ताक्षर प्रमाणीकरण, सामग्री अखण्डता, र प्रस्तुत गरिएको अघिल्लो लिंकलाई प्रमाणित गर्छन्। यसले कार्यान्वयन, स्रोत सत्य, विश्वासयुक्त समय, पूर्णता, नीति मूल्याङ्कन, वा सहि रूपमा प्रमाणित गर्दैन। रसिदहरू एक आधार हुन्। सरकार प्रणाली तपाईंले माथि निर्माण गर्नुहुने हो।

त्यो सीमालाई ध्यानमा राख्दै पाठ पुस्तकको README फेरि पढ्नुहोस्। सबैभन्दा सामान्य गल्ती टोलीहरूले रसिदसँग "हामीसँग रसिदहरू छन्" भन्नाले "हामी शासित छौं" भन्ने बुझ्नु हो। यो होइन। रसिदहरूले एजेन्ट व्यवहारले अडिटयोग्य बनाउँछन्, सहि बनाउँदैनन्।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
